# EU27 Industrial Energy Consumption — 2000 to 2023
### JRC-IDEES 2023 — Process-Level Final Energy Consumption

---

## Scope

This notebook builds a clean, long-format dataset of EU27 industrial final energy consumption (FEC)
from the **JRC-IDEES 2023** database (published November 2025). It covers all 27 Member States,
11 industrial sectors, and provides a granular breakdown by process and fuel type over 2000–2023.

The output Parquet file feeds directly into the industry heat dashboard.

**Source:** https://data.jrc.ec.europa.eu/dataset/jrc-idees-2023  
**Unit:** ktoe (converted to TWh for the dashboard)  
**Coverage:** EU27 Member States · 11 sectors · process × fuel × year  


---
## Setup — Imports

In [4]:
import pandas as pd
from pathlib import Path

---
## Part 1 — JRC-IDEES 2023
### Process-Level Energy Consumption by Fuel Type (2000–2023)

The JRC-IDEES 2023 dataset provides a much more granular breakdown than the Zenodo dataset — down to individual industrial **processes** and **fuel types** within each sector. This allows us to understand not just *how much* energy is consumed, but *how* it is delivered and *what* fuels could be replaced by renewables.

Each Excel file contains one country, with one sheet per sector (e.g. `ISI_fec` for Iron & Steel). Each sheet uses a structured code system:

```
FEC . ktoe . AT . ISI . BF_BOF . BLAST_FURNACE . THERM . COKE
             ↑    ↑       ↑            ↑            ↑      ↑
           country sector subsector  process    end_use  fuel
```

### 1.1 — Parser Function

We define a reusable function that loads one `_fec` sheet, parses the structured code column into separate dimensions, filters out aggregate totals, and melts the year columns into long format.

In [8]:
def parse_fec_sheet(file_path, sheet_name):
    """
    Loads and parses a _fec sheet from a JRC-IDEES Industry Excel file.
    
    Parameters
    ----------
    file_path : Path
        Path to the country Excel file (e.g. JRC-IDEES-2023_Industry_AT.xlsx)
    sheet_name : str
        Name of the sheet to parse (e.g. 'ISI_fec')
    
    Returns
    -------
    DataFrame with columns:
        country, sector, subsector, process, end_use_type, fuel, year, value_ktoe
    """
    df = pd.read_excel(file_path, sheet_name=sheet_name)

    code_col = "Code"
    year_cols = list(range(2000, 2024))

    # Keep only rows that have a structured code — removes headers and section titles
    df = df[df[code_col].notna()].copy()

    # Split the code into its 8 components
    code_parts = df[code_col].str.split(".", expand=True)
    code_parts.columns = ["metric", "unit", "country", "sector",
                          "subsector", "process", "end_use_type", "fuel"]
    code_parts.index = df.index   # align index to avoid concat mismatches

    # Attach parsed code columns to the main DataFrame
    df = pd.concat([df, code_parts], axis=1)

    # Remove aggregate TOTAL rows to avoid double counting
    df = df[
        (df["fuel"] != "TOTAL") &
        (df["process"] != "TOTAL")
    ]

    # Keep only the columns we need
    cols_to_keep = ["country", "sector", "subsector", "process",
                    "end_use_type", "fuel"] + year_cols
    df = df[cols_to_keep]

    # Melt year columns into long format
    df_long = df.melt(
        id_vars=["country", "sector", "subsector", "process",
                 "end_use_type", "fuel"],
        var_name="year",
        value_name="value_ktoe"
    )

    return df_long

### 1.2 — Test the Parser on Two Sectors

Before running on all 27 countries, we test the function on Iron & Steel and Food & Beverages for Austria to confirm the output is correct.

In [9]:
file_path = Path("JRC-IDEES-2023/AT/JRC-IDEES-2023_Industry_AT.xlsx")

df_test_isi = parse_fec_sheet(file_path, "ISI_fec")
df_test_fbt = parse_fec_sheet(file_path, "FBT_fec")

print(f"ISI_fec shape: {df_test_isi.shape}")
print(f"FBT_fec shape: {df_test_fbt.shape}")

print("\nISI_fec sample:")
print(df_test_isi.head())

print("\nFBT_fec sample:")
print(df_test_fbt.head())

ISI_fec shape: (1704, 8)
FBT_fec shape: (1536, 8)

ISI_fec sample:
  country sector subsector   process end_use_type           fuel  year  \
0      AT    ISI    BF_BOF     LIGHT      GENERIC           ELEC  2000   
1      AT    ISI    BF_BOF   AIRCOMP      GENERIC           ELEC  2000   
2      AT    ISI    BF_BOF     MOTOR      GENERIC           ELEC  2000   
3      AT    ISI    BF_BOF      FANS      GENERIC           ELEC  2000   
4      AT    ISI    BF_BOF  LOW_ENTH        THERM  DIESEL_LIQBIO  2000   

   value_ktoe  
0    1.783841  
1    0.951382  
2   23.784541  
3    0.594614  
4    0.000000  

FBT_fec sample:
  country sector subsector   process end_use_type           fuel  year  \
0      AT    FBT       FBT     LIGHT      GENERIC           ELEC  2000   
1      AT    FBT       FBT   AIRCOMP      GENERIC           ELEC  2000   
2      AT    FBT       FBT     MOTOR      GENERIC           ELEC  2000   
3      AT    FBT       FBT      FANS      GENERIC           ELEC  2000   
4    

### 1.3 — Build the Full Austria Dataset

Once the parser is validated, we run it on all 11 industrial sectors for Austria and stack the results.

In [10]:
fec_sheets = ["ISI_fec", "NFM_fec", "CHI_fec", "NMM_fec", "PPA_fec",
              "FBT_fec", "TRE_fec", "MAE_fec", "TEL_fec", "WWP_fec", "OIS_fec"]

all_sectors = []
for sheet in fec_sheets:
    print(f"Parsing {sheet}...")
    df_sector = parse_fec_sheet(file_path, sheet)
    all_sectors.append(df_sector)

df_austria = pd.concat(all_sectors, ignore_index=True)

print(f"\nAustria dataset shape: {df_austria.shape}")
print(f"Sectors: {df_austria['sector'].unique()}")
df_austria.head()

Parsing ISI_fec...
Parsing NFM_fec...
Parsing CHI_fec...
Parsing NMM_fec...
Parsing PPA_fec...
Parsing FBT_fec...
Parsing TRE_fec...
Parsing MAE_fec...
Parsing TEL_fec...
Parsing WWP_fec...
Parsing OIS_fec...

Austria dataset shape: (18096, 8)
Sectors: ['ISI' 'NFM' 'CHI' 'NMM' 'PPA' 'FBT' 'TRE' 'MAE' 'TEL' 'WWP' 'OIS']


,country,sector,subsector,process,end_use_type,fuel,year,value_ktoe
0,AT,ISI,BF_BOF,LIGHT,GENERIC,ELEC,2000,1.783841
1,AT,ISI,BF_BOF,AIRCOMP,GENERIC,ELEC,2000,0.951382
2,AT,ISI,BF_BOF,MOTOR,GENERIC,ELEC,2000,23.784541
3,AT,ISI,BF_BOF,FANS,GENERIC,ELEC,2000,0.594614
4,AT,ISI,BF_BOF,LOW_ENTH,THERM,DIESEL_LIQBIO,2000,0.000000


### 1.4 — Build the Full EU27 Dataset

We now loop over all 27 country folders, parse all 11 sectors for each country, and stack everything into one master dataset.

In [11]:
base_folder = Path("JRC-IDEES-2023")

all_countries = []

for country_folder in sorted(base_folder.iterdir()):
    if not country_folder.is_dir():
        continue

    country_code = country_folder.name
    file_path = country_folder / f"JRC-IDEES-2023_Industry_{country_code}.xlsx"

    if not file_path.exists():
        print(f"Skipping {country_code} — Industry file not found")
        continue

    print(f"Processing {country_code}...")

    country_sectors = []
    for sheet in fec_sheets:
        df_sector = parse_fec_sheet(file_path, sheet)
        country_sectors.append(df_sector)

    df_country = pd.concat(country_sectors, ignore_index=True)
    all_countries.append(df_country)

df_eu27 = pd.concat(all_countries, ignore_index=True)

print(f"\nFull EU27 dataset shape: {df_eu27.shape}")
print(f"Countries loaded: {df_eu27['country'].unique()}")

Processing AT...
Processing BE...
Processing BG...
Processing CY...
Processing CZ...
Processing DE...
Processing DK...
Processing EE...
Processing EL...
Processing ES...
Processing EU27...
Processing FI...
Processing FR...
Processing HR...
Processing HU...
Processing IE...
Processing IT...
Processing LT...
Processing LU...
Processing LV...
Processing MT...
Processing NL...
Processing PL...
Processing PT...
Processing RO...
Processing SE...
Processing SI...
Processing SK...

Full EU27 dataset shape: (506688, 8)
Countries loaded: ['AT' 'BE' 'BG' 'CY' 'CZ' 'DE' 'DK' 'EE' 'EL' 'ES' 'EU27' 'FI' 'FR' 'HR'
 'HU' 'IE' 'IT' 'LT' 'LU' 'LV' 'MT' 'NL' 'PL' 'PT' 'RO' 'SE' 'SI' 'SK']


### 1.5 — Convert Energy Values to TWh

The JRC-IDEES dataset reports energy in ktoe (kilotonne of oil equivalent), the standard Eurostat unit for historical energy balances. We add a TWh column for compatibility with the Zenodo dataset and modern policy reporting (1 ktoe = 0.01163 TWh).

In [13]:
#Convert value_ktoe to TWh (1 ktoe = 0.01163 TWh)
df_eu27["value_twh"] = df_eu27["value_ktoe"] * 0.01163

---
## Part 2 — Data Quality Checks

Before applying any labels or saving the dataset, we run a series of validation checks to catch any parsing errors or unexpected patterns.

### 2.1 — Remove EU27 Aggregate

The EU27 folder contains an aggregate file that sums all 27 countries. We remove it to avoid double counting in any analysis.

In [14]:
df_eu27_clean = df_eu27[df_eu27["country"] != "EU27"].copy()
print(f"Shape after removing EU27 aggregate: {df_eu27_clean.shape}")

Shape after removing EU27 aggregate: (488592, 9)


### 2.2 — Row Count Consistency

Every country should have exactly the same number of rows — the JRC uses a standardised template across all Member States.

In [15]:
row_counts = df_eu27_clean.groupby("country").size().sort_values()
print("Row counts per country:")
print(row_counts)
print(f"\nAll equal: {row_counts.nunique() == 1}")

Row counts per country:
country
AT    18096
SE    18096
RO    18096
PT    18096
PL    18096
NL    18096
MT    18096
LV    18096
LU    18096
LT    18096
IT    18096
IE    18096
SI    18096
HU    18096
FR    18096
FI    18096
ES    18096
EL    18096
EE    18096
DK    18096
DE    18096
CZ    18096
CY    18096
BG    18096
BE    18096
HR    18096
SK    18096
dtype: int64

All equal: True


### 2.3 — Missing Values

In [16]:
print("Missing values per column:")
print(df_eu27_clean.isnull().sum())

Missing values per column:
country         0
sector          0
subsector       0
process         0
end_use_type    0
fuel            0
year            0
value_ktoe      0
value_twh       0
dtype: int64


### 2.4 — Value Range Sanity Check

Energy consumption cannot be negative. We also check the overall distribution.

In [17]:
print(f"Negative values: {(df_eu27_clean['value_ktoe'] < 0).sum()}")
print("\nValue distribution (ktoe):")
print(df_eu27_clean["value_ktoe"].describe())

print("\nMaximum value row:")
print(df_eu27_clean.loc[df_eu27_clean["value_ktoe"].idxmax()])

Negative values: 0

Value distribution (ktoe):
count    488592.000000
mean         12.240090
std          67.902621
min           0.000000
25%           0.000000
50%           0.107383
75%           2.733891
max        2748.996389
Name: value_ktoe, dtype: float64

Maximum value row:
country                    SE
sector                    PPA
subsector                PULP
process               PULPING
end_use_type            STEAM
fuel            BIOMASS_WASTE
year                     2023
value_ktoe        2748.996389
value_twh           31.970828
Name: 463840, dtype: object


### 2.5 — Cross-Check Against Known Totals

We convert the 2023 total to TWh and compare it against the Zenodo dataset (2021 baseline: ~2,315 TWh). We also verify the country ranking matches expectations — Germany should dominate, followed by Italy, France, Spain, Poland.

In [18]:
# Total EU27 FEC in 2023 (1 ktoe = 0.01163 TWh)
total_2023_twh = (
    df_eu27_clean[df_eu27_clean["year"] == 2023]["value_ktoe"].sum() * 0.01163
)
print(f"Total EU27 industry FEC 2023: {total_2023_twh:.1f} TWh")
print("(Expected: ~2,300–2,600 TWh based on Zenodo 2021 baseline)")

print("\nCountry totals 2023 (TWh):")
country_totals = (
    df_eu27_clean[df_eu27_clean["year"] == 2023]
    .groupby("country")["value_ktoe"]
    .sum()
    .sort_values(ascending=False)
    * 0.01163
)
print(country_totals.round(1))

Total EU27 industry FEC 2023: 2497.3 TWh
(Expected: ~2,300–2,600 TWh based on Zenodo 2021 baseline)

Country totals 2023 (TWh):
country
DE    582.3
FR    278.6
IT    273.3
ES    214.9
PL    159.6
NL    135.3
SE    132.0
FI    108.1
BE    107.9
AT     80.7
CZ     69.3
RO     57.7
PT     50.2
HU     46.5
SK     33.4
EL     28.6
BG     28.5
DK     25.9
IE     24.0
HR     13.7
SI     12.5
LV     10.7
LT     10.5
LU      5.7
EE      3.9
CY      2.9
MT      0.7
Name: value_ktoe, dtype: float64


---
## Part 3 — Label Mapping

The dataset currently uses JRC internal codes (e.g. `ISI`, `BF_BOF`, `THERM`). We replace these with human-readable labels for use in the dashboard.

**Important:** EU27 must be removed before mapping — it has no entry in the country dictionary and would produce NaN values.

In [19]:
# --- Label dictionaries ---

country_names = {
    "AT": "Austria", "BE": "Belgium", "BG": "Bulgaria", "CY": "Cyprus",
    "CZ": "Czech Republic", "DE": "Germany", "DK": "Denmark", "EE": "Estonia",
    "EL": "Greece", "ES": "Spain", "FI": "Finland", "FR": "France",
    "HR": "Croatia", "HU": "Hungary", "IE": "Ireland", "IT": "Italy",
    "LT": "Lithuania", "LU": "Luxembourg", "LV": "Latvia", "MT": "Malta",
    "NL": "Netherlands", "PL": "Poland", "PT": "Portugal", "RO": "Romania",
    "SE": "Sweden", "SI": "Slovenia", "SK": "Slovakia"
}

sector_names = {
    "ISI": "Iron & Steel",
    "NFM": "Non-ferrous Metals",
    "CHI": "Chemicals",
    "NMM": "Non-metallic Minerals",
    "PPA": "Pulp, Paper & Printing",
    "FBT": "Food, Beverages & Tobacco",
    "TRE": "Transport Equipment",
    "MAE": "Machinery & Equipment",
    "TEL": "Textiles & Leather",
    "WWP": "Wood & Wood Products",
    "OIS": "Other Industrial Sectors"
}

subsector_names = {
    "BF_BOF": "Blast Furnace & Basic Oxygen Furnace",
    "EAF": "Electric Arc Furnace",
    "ALUMINA": "Alumina Production",
    "PRIM_ALU": "Primary Aluminium",
    "SEC_ALU": "Secondary Aluminium",
    "OTHER_NFM": "Other Non-ferrous Metals",
    "BASIC_CHEM": "Basic Chemicals",
    "OTHER_CHEM": "Other Chemicals",
    "PHARM": "Pharmaceuticals",
    "CEM": "Cement",
    "CER": "Ceramics",
    "GLASS": "Glass",
    "PULP": "Pulp",
    "PAPER": "Paper",
    "PRINT": "Printing",
    "FBT": "Food, Beverages & Tobacco",
    "TRE": "Transport Equipment",
    "MAE": "Machinery & Equipment",
    "TEL": "Textiles & Leather",
    "WWP": "Wood & Wood Products",
    "OIS": "Other Industrial Sectors"
}

process_names = {
    "LIGHT": "Lighting",
    "AIRCOMP": "Air Compressors",
    "MOTOR": "Motor Drives",
    "FANS": "Fans & Pumps",
    "LOW_ENTH": "Low-enthalpy Heat",
    "SINTERING": "Sintering & Pellet-making",
    "BLAST_FURNACE": "Blast Furnace",
    "REFINING": "Furnaces, Refining & Rolling",
    "FINISHING": "Product Finishing",
    "FINISHING_STEAM": "Product Finishing (Steam)",
    "SMELTING": "Smelting",
    "ARC_FURNACE": "Electric Arc Furnace",
    "A_PRODUCTION": "Aluminium Production",
    "A_REFINING": "Aluminium Refining",
    "PROCESSING": "Processing",
    "SEC_REMELTING": "Secondary Remelting",
    "SEC_PROCESSING": "Secondary Processing",
    "SEC_FINISHING": "Secondary Finishing",
    "SEC_FINISHING_STEAM": "Secondary Finishing (Steam)",
    "OTH_PRODUCTION": "Other Production",
    "OTH_PROCESSING": "Other Processing",
    "OTH_FINISHING": "Other Finishing",
    "OTH_FINISHING_STEAM": "Other Finishing (Steam)",
    "PROC_HEAT": "Process Heat",
    "PROC_COOL": "Process Cooling",
    "PROC_COOL_STEAM": "Process Cooling (Steam)",
    "GENERIC": "Generic Process",
    "PROCESSING_STEAM": "Processing (Steam)",
    "GRINDING_RAW": "Grinding Raw Materials",
    "PREHEAT": "Pre-heating",
    "KILN": "Kiln",
    "PRECAST": "Pre-casting",
    "PRECAST_STEAM": "Pre-casting (Steam)",
    "MIXING": "Mixing",
    "DRYING": "Drying",
    "DRYING_STEAM": "Drying (Steam)",
    "DRYING_MICROW": "Drying (Microwave)",
    "KILN_THERM": "Kiln (Thermal)",
    "KILN_ELEC": "Kiln (Electric)",
    "FINISHING_ELEC": "Finishing (Electric)",
    "MELTING": "Melting",
    "MELTING_ELEC": "Melting (Electric)",
    "FORMING": "Forming",
    "ANNEALING": "Annealing",
    "ANNEALING_ELEC": "Annealing (Electric)",
    "PREPARATION": "Preparation",
    "PULPING": "Pulping",
    "PULPING_MECH": "Pulping (Mechanical)",
    "BLEACHING": "Bleaching",
    "PREPARATION_ELEC": "Preparation (Electric)",
    "PAPER_MACHINE": "Paper Machine",
    "PAPER_MACHINE_ELEC": "Paper Machine (Electric)",
    "PRINTING": "Printing",
    "OVEN": "Oven (Direct Heat)",
    "OVEN_MICROW": "Oven (Microwave)",
    "PROC_HEAT_MICROW": "Process Heat (Microwave)",
    "DRYING_ELEC": "Drying (Electric)",
    "DRYING_FREEZE": "Freeze Drying",
    "PROC_COOL_THERM": "Process Cooling (Thermal)",
    "PROC_COOL_ELEC": "Process Cooling (Electric)",
    "FOUNDRY": "Foundry",
    "CONNECTION_THERM": "Connection (Thermal)",
    "CONNECTION_ELEC": "Connection (Electric)",
    "HEAT_TREAT": "Heat Treatment",
    "PRETREAT": "Pre-treatment",
    "PROC_HEAT_ELEC": "Process Heat (Electric)"
}

end_use_type_names = {
    "GENERIC": "Electricity Service",
    "THERM": "Thermal Heat",
    "HP": "Heat Pump",
    "ELEC": "Electric Process",
    "STEAM": "Steam",
    "MECH": "Mechanical",
    "MICROW": "Microwave",
    "FREEZE_DRY": "Freeze Drying"
}

fuel_names = {
    "ELEC": "Electricity",
    "DIESEL_LIQBIO": "Diesel & Liquid Biofuels",
    "NG_BIOGAS": "Natural Gas & Biogas",
    "SOLAR_GEO": "Solar & Geothermal",
    "AMBIENT": "Ambient Heat",
    "NONCOKE_SOLIDS": "Non-coke Solids",
    "RFO": "Residual Fuel Oil",
    "DERIVED": "Derived Gases",
    "COKE": "Coke",
    "LPG": "LPG",
    "RFG": "Refinery Gas",
    "OTHER": "Other Liquids",
    "BIOMASS_WASTE": "Biomass & Waste",
    "STEAM_DISTR": "Distributed Steam",
    "SOLIDS": "Solids"
}

# --- Apply all mappings ---
df_eu27_clean["country"]      = df_eu27_clean["country"].map(country_names)
df_eu27_clean["sector"]       = df_eu27_clean["sector"].map(sector_names)
df_eu27_clean["subsector"]    = df_eu27_clean["subsector"].map(subsector_names)
df_eu27_clean["process"]      = df_eu27_clean["process"].map(process_names)
df_eu27_clean["end_use_type"] = df_eu27_clean["end_use_type"].map(end_use_type_names)
df_eu27_clean["fuel"]         = df_eu27_clean["fuel"].map(fuel_names)

print("Unmapped values per column (should all be 0):")
for col in ["country", "sector", "subsector", "process", "end_use_type", "fuel"]:
    missing = df_eu27_clean[col].isna().sum()
    print(f"  {col}: {missing}")

Unmapped values per column (should all be 0):
  country: 0
  sector: 0
  subsector: 0
  process: 0
  end_use_type: 0
  fuel: 0


### 3.1 — Final Dataset Preview

In [20]:
print(f"Final dataset shape: {df_eu27_clean.shape}")
print(f"\nColumns: {df_eu27_clean.columns.tolist()}")
print(f"\nYears covered: {df_eu27_clean['year'].min()} – {df_eu27_clean['year'].max()}")
print(f"Countries: {df_eu27_clean['country'].nunique()}")
print(f"Sectors: {df_eu27_clean['sector'].nunique()}")
df_eu27_clean.head(10)

Final dataset shape: (488592, 9)

Columns: ['country', 'sector', 'subsector', 'process', 'end_use_type', 'fuel', 'year', 'value_ktoe', 'value_twh']

Years covered: 2000 – 2023
Countries: 27
Sectors: 11


,country,sector,subsector,process,end_use_type,fuel,year,value_ktoe,value_twh
0,Austria,Iron & Steel,Blast Furnace & Basic Oxygen Furnace,Lighting,Electricity Service,Electricity,2000,1.783841,0.020746
1,Austria,Iron & Steel,Blast Furnace & Basic Oxygen Furnace,Air Compressors,Electricity Service,Electricity,2000,0.951382,0.011065
2,Austria,Iron & Steel,Blast Furnace & Basic Oxygen Furnace,Motor Drives,Electricity Service,Electricity,2000,23.784541,0.276614
3,Austria,Iron & Steel,Blast Furnace & Basic Oxygen Furnace,Fans & Pumps,Electricity Service,Electricity,2000,0.594614,0.006915
4,Austria,Iron & Steel,Blast Furnace & Basic Oxygen Furnace,Low-enthalpy Heat,Thermal Heat,Diesel & Liquid Biofuels,2000,0.000000,0.000000
5,Austria,Iron & Steel,Blast Furnace & Basic Oxygen Furnace,Low-enthalpy Heat,Thermal Heat,Natural Gas & Biogas,2000,0.644637,0.007497
6,Austria,Iron & Steel,Blast Furnace & Basic Oxygen Furnace,Low-enthalpy Heat,Thermal Heat,Solar & Geothermal,2000,0.000000,0.000000
7,Austria,Iron & Steel,Blast Furnace & Basic Oxygen Furnace,Low-enthalpy Heat,Heat Pump,Ambient Heat,2000,0.000000,0.000000
8,Austria,Iron & Steel,Blast Furnace & Basic Oxygen Furnace,Low-enthalpy Heat,Thermal Heat,Electricity,2000,0.540762,0.006289
9,Austria,Iron & Steel,Blast Furnace & Basic Oxygen Furnace,Sintering & Pellet-making,Thermal Heat,Non-coke Solids,2000,0.000000,0.000000


---
## Part 4 — Save the Dataset

We save the final dataset as a Parquet file — faster to load and smaller than CSV, ideal for feeding into the dashboard.

In [21]:
output_path = Path("eu27_industry_fec_2000_2023.parquet")
df_eu27_clean.to_parquet(output_path, index=False)
print(f"Dataset saved to: {output_path}")
print(f"File size: {output_path.stat().st_size / 1024 / 1024:.1f} MB")

Dataset saved to: eu27_industry_fec_2000_2023.parquet
File size: 5.6 MB


In [22]:
output_path = Path("/Users/francescocalzati/Desktop/industry_heat_eu/industry_heat_dashboard/data/eu27_industry_fec_2000_2023.parquet")
df_eu27_clean.to_parquet(output_path, index=False)
print("Saved.")

Saved.
